# 83장 · 도구 호출 경로(Trajectory) 평가

승인을 받기 전에 확정 도구를 불러도 마지막 답변은 “처리했습니다”일 수 있습니다. 답변 문구와 별도로 요청 ID, 승인 시점, 실제 저장 건수를 확인합니다.

AgentEvals와 사용자 정의 검사로 도구 선택·인자·선후 관계·중복 호출을 검증한다. 순서를 바꿔도 되는 조회와 승인 후 수행할 확정 작업을 구분하고, 실제 처리 횟수는 앱 기록과 대조한다.

## 도구 인자가 같은 요청을 가리키나요?

In [ ]:
expected_id = "T07"
events = [("lookup_user", "T07"), ("lookup_policy", "T07"), ("approve", "T07"), ("commit", "T08")]
def valid_path(events, request_id, writes):
    approved = False
    commits = 0
    lookups = set()
    for name, target in events:
        if target != request_id:
            return False
        if name not in {"lookup_user", "lookup_policy", "approve", "commit"}:
            return False
        if name.startswith("lookup_"):
            lookups.add(name)
        if name == "approve":
            if lookups != {"lookup_user", "lookup_policy"}:
                return False
            approved = True
        if name == "commit":
            if not approved:
                return False
            commits += 1
    return commits == 1 and writes == [request_id]

print(valid_path(events, expected_id, ["T08"]))

False


승인된 대상은 T07인데 마지막 도구의 인자와 저장 기록은 T08입니다. 승인 이벤트는 앱이 사람의 확인을 기록한 값입니다. 모델이 승인했다고 말한 내용은 여기에 넣지 않습니다.

## 승인 경로와 실제 변경 횟수 함께 검사하기

82장의 검색 후보를 기준선으로 두고 80장의 변경 기록도 함께 검사합니다. 조회한 항목과 저장한 항목이 다른데도 답변만 보면 “완료”처럼 보일 수 있습니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/evaluation/ranking.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/evaluation/ranking.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'case_id': 'july-deadline',
     'as_of': '2026-07-01',
     'ranked': ['A1', 'A2'],
     'observed': 30,
     'expected': 90,
     'gold': 'A2',
     'mode': 'local',
     'editions': [['A1', '2026-06-01', 30], ['A2', '2026-07-01', 90]],
     'approval': {'item_id': 2,
                  'statuses': [200, 409],
                  'version': 1,
                  'done': True,
                  'mode': 'local-version-test',
                  'events': [['read', 2], ['approve', 2], ['write', 2]],
                  'writes': [2]},
     'release_version': 'edition-sort-v2',
     'before': {'recall': 1.0, 'mrr': 0.5},
     'after': {'recall': 1.0, 'mrr': 1.0},
     'candidate': {'citation': 'A2', 'days': 90},
     'candidate_rank': ['A2', 'A1'],
     'candidate_editions': [['A2', '2026-07-01', 90], ['A1', '2026-06-01', 30]]}
    print("복귀 예시", incoming.name)

내 실습 파일 ranking.json


### 승인 기록과 실제 변경 건수 비교하기

In [ ]:
approval = previous["approval"]
row_id = approval["item_id"]
def accepted(events, writes):
    expected = [("read", row_id), ("approve", row_id), ("write", row_id)]
    return events == expected and writes == [row_id]
events = [tuple(event) for event in approval["events"]]
writes = approval["writes"]
path_ok = accepted(events, writes)
result = {**previous, "path_ok": path_ok, "write_count": len(writes), "events": events}
print(path_ok, len(writes))

True 1


events에는 80장의 함수가 실제로 남긴 사건이, writes에는 항목을 변경할 때 기록한 ID가 들어 있습니다. 실제 서비스에서는 같은 요청 ID의 DB 변경 로그로 writes를 채워야 합니다. 아래 AgentEvals 예제에서 도구 이름과 인자도 비교하세요.

### 직접 해 보기

이벤트는 그대로 두고 실제 저장 목록에 같은 ID를 하나 더 넣으세요. 한 번만 저장했다는 조건이 깨지는지 확인하세요.

In [ ]:
# accepted의 두 번째 인자만 바꿔 보세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>print(accepted(events, writes + [row_id]))</code></pre><pre>False</pre><p>같은 요청이 두 번 저장되면 사건 순서가 맞아도 실패합니다.</p></details>

### 84장에서 다시 열 파일

방금 계산한 값을 service-lab/evaluation/trajectory.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "evaluation/trajectory.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

trajectory.json


### 직접 해 보기

독립적인 두 조회 순서만 바꾼 경우, 승인 전에 확정한 경우, 같은 요청을 두 번 저장한 경우를 각각 검사하세요. 마지막 사례에서는 호출 기록은 한 번으로 두세요.

In [ ]:
# valid_path에 실행 사건과 실제 저장 ID 목록을 각각 전달하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>lookups = [(&quot;lookup_user&quot;, &quot;T07&quot;), (&quot;lookup_policy&quot;, &quot;T07&quot;)]
approval, commit = (&quot;approve&quot;, &quot;T07&quot;), (&quot;commit&quot;, &quot;T07&quot;)
print(valid_path(lookups[::-1] + [approval, commit], &quot;T07&quot;, [&quot;T07&quot;]))
print(valid_path(lookups + [commit, approval], &quot;T07&quot;, [&quot;T07&quot;]))
print(valid_path(lookups + [approval, commit], &quot;T07&quot;, [&quot;T07&quot;, &quot;T07&quot;]))</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>True
False
False
</code></pre></details>

## 수업 서비스에 적용하기

agentevals를 설치한 커널에서 아래 셀을 실행하세요. 같은 도구에 다른 요청 ID를 넣었을 때 strict 평가 점수가 어떻게 바뀌는지 확인합니다. 팀 서비스에서는 조회 순서의 교환을 허용할 부분과 승인이 앞서야 할 부분을 나눕니다. LangSmith 메시지에서 읽은 호출과 앱의 저장 로그를 요청 ID로 연결해 두 결과를 제출하세요.

In [ ]:
run_integration = False
if run_integration:
    import json
    from agentevals.trajectory.match import create_trajectory_match_evaluator
    def trajectory(ticket):
        return [{"role": "assistant", "content": "", "tool_calls": [
            {"id":"call1", "type":"function", "function":{
                "name":"lookup", "arguments":json.dumps({"ticket":ticket})}}]}]
    check = create_trajectory_match_evaluator(trajectory_match_mode="strict",
                                              tool_args_match_mode="exact")
    for ticket in ["T07", "T08"]:
        print(ticket, check(outputs=trajectory(ticket), reference_outputs=trajectory("T07"))["score"])

### 제출 전 확인

각 요청에서 승인 여부와 실제로 바뀐 항목 수를 대조하세요.

[공식 문서](https://github.com/langchain-ai/agentevals)